In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch

from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    confusion_matrix,
)

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_utils import load_mnist_vae_data, get_device, seed_everything
from src.models import ConvVAE
from src.latent_analysis import encode_dataset

In [ ]:
seed_everything(42)
device = get_device()
device

In [ ]:
best_config = {
    "latent_dim": 16,
    "beta": 1.0,
    "epochs": 20,
    "lr": 1e-3,
    "hidden_dim": 256,
    "base_channels": 64,
    "batch_size": 512,
    "warmup_epochs": None,
    "recon_loss_type": "bce",
}
best_config

In [ ]:
train_loader, val_loader, test_loader = load_mnist_vae_data(
    data_dir="../data",
    batch_size=best_config["batch_size"],
    val_ratio=0.1,
    num_workers=0,
    random_state=42,
)

In [ ]:
vae_ckpt_path = "../results/logs/h1_conv_vae_best_latent16.pt"

vae16 = ConvVAE(
    input_channels=1,
    latent_dim=16,
    hidden_dim=best_config["hidden_dim"],
    base_channels=best_config["base_channels"],
).to(device)

state_dict = torch.load(vae_ckpt_path, map_location=device)
vae16.load_state_dict(state_dict)
vae16.eval()

print(f"Loaded: {vae_ckpt_path}")

In [ ]:
def loader_to_numpy(data_loader):
    xs, ys = [], []
    for x, y in data_loader:
        xs.append(x.view(x.size(0), -1).numpy())
        ys.append(y.numpy())
    X = np.concatenate(xs, axis=0)
    y = np.concatenate(ys, axis=0)
    return X, y

X_train_raw, y_train = loader_to_numpy(train_loader)
X_test_raw, y_test = loader_to_numpy(test_loader)

print("X_train_raw:", X_train_raw.shape)
print("X_test_raw :", X_test_raw.shape)

In [ ]:
pca16 = PCA(n_components=16, random_state=42)
Z_train_pca16 = pca16.fit_transform(X_train_raw)
Z_test_pca16 = pca16.transform(X_test_raw)

print("PCA-16 explained variance ratio sum:", pca16.explained_variance_ratio_.sum())
print("Z_train_pca16:", Z_train_pca16.shape)
print("Z_test_pca16 :", Z_test_pca16.shape)

In [ ]:
encoded_train_vae = encode_dataset(vae16, train_loader, device=device, use_mu=True)
encoded_test_vae = encode_dataset(vae16, test_loader, device=device, use_mu=True)

Z_train_vae16 = encoded_train_vae["mu"]
Z_test_vae16 = encoded_test_vae["mu"]

print("Z_train_vae16:", Z_train_vae16.shape)
print("Z_test_vae16 :", Z_test_vae16.shape)

In [ ]:
def clustering_accuracy_via_majority(y_true, y_pred):
    """
    不是最优匹配 accuracy，而是多数投票 purity 风格映射。
    H3 里更建议把 purity 作为主指标之一，而不是叫 accuracy。
    """
    cm = confusion_matrix(y_true, y_pred)
    return np.sum(np.max(cm, axis=0)) / np.sum(cm)


def purity_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    return np.sum(np.max(cm, axis=0)) / np.sum(cm)


def evaluate_clustering(y_true, y_pred):
    return {
        "ARI": adjusted_rand_score(y_true, y_pred),
        "NMI": normalized_mutual_info_score(y_true, y_pred),
        "Purity": purity_score(y_true, y_pred),
    }

In [ ]:
def run_kmeans_eval(X_train, X_test, y_test, n_clusters=10, random_state=42):
    model = KMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        n_init=20,
    )
    model.fit(X_train)
    y_pred = model.predict(X_test)
    metrics = evaluate_clustering(y_test, y_pred)
    return model, y_pred, metrics


def run_gmm_eval(X_train, X_test, y_test, n_components=10, random_state=42):
    model = GaussianMixture(
        n_components=n_components,
        covariance_type="diag",
        random_state=random_state,
        reg_covar=1e-6,
        n_init=5,
    )
    model.fit(X_train)
    y_pred = model.predict(X_test)
    metrics = evaluate_clustering(y_test, y_pred)
    return model, y_pred, metrics

In [ ]:
representations = {
    "Raw-784": (X_train_raw, X_test_raw),
    "PCA-16": (Z_train_pca16, Z_test_pca16),
    "VAE-16(mu)": (Z_train_vae16, Z_test_vae16),
}

In [ ]:
results = []

pred_store = {}

for rep_name, (Xtr, Xte) in representations.items():
    # KMeans
    km_model, km_pred, km_metrics = run_kmeans_eval(
        Xtr, Xte, y_test, n_clusters=10, random_state=42
    )
    results.append({
        "representation": rep_name,
        "method": "KMeans",
        **km_metrics
    })
    pred_store[(rep_name, "KMeans")] = km_pred

    # GMM
    gmm_model, gmm_pred, gmm_metrics = run_gmm_eval(
        Xtr, Xte, y_test, n_components=10, random_state=42
    )
    results.append({
        "representation": rep_name,
        "method": "GMM",
        **gmm_metrics
    })
    pred_store[(rep_name, "GMM")] = gmm_pred

results_df = pd.DataFrame(results).sort_values(["method", "ARI"], ascending=[True, False])
results_df

In [ ]:
os.makedirs("../results/tables", exist_ok=True)
results_df.to_csv("../results/tables/h3_latent_clustering_metrics.csv", index=False)
results_df

In [ ]:
os.makedirs("../results/figures/h3_latent_clustering", exist_ok=True)

In [ ]:
def plot_metric_bar(results_df, method, metric, save_path=None):
    sub = results_df[results_df["method"] == method].copy()

    plt.figure(figsize=(7, 4))
    plt.bar(sub["representation"], sub[metric])
    plt.ylim(0, 1.0)
    plt.ylabel(metric)
    plt.title(f"{metric} comparison under {method}")
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()

In [ ]:
for method in ["KMeans", "GMM"]:
    for metric in ["ARI", "NMI", "Purity"]:
        plot_metric_bar(
            results_df,
            method=method,
            metric=metric,
            save_path=f"../results/figures/h3_latent_clustering/{method.lower()}_{metric.lower()}.png"
        )

In [ ]:
def plot_grouped_metric(results_df, metric, save_path=None):
    pivot_df = results_df.pivot(index="representation", columns="method", values=metric)

    ax = pivot_df.plot(kind="bar", figsize=(8, 4), rot=0)
    ax.set_ylim(0, 1.0)
    ax.set_ylabel(metric)
    ax.set_title(f"{metric}: representation comparison")
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()

In [ ]:
for metric in ["ARI", "NMI", "Purity"]:
    plot_grouped_metric(
        results_df,
        metric=metric,
        save_path=f"../results/figures/h3_latent_clustering/grouped_{metric.lower()}.png"
    )

In [ ]:
best_row = results_df.sort_values("ARI", ascending=False).iloc[0]
best_row

In [ ]:
best_rep = best_row["representation"]
best_method = best_row["method"]
best_pred = pred_store[(best_rep, best_method)]

cm = confusion_matrix(y_test, best_pred)

plt.figure(figsize=(7, 6))
plt.imshow(cm, aspect="auto")
plt.colorbar()
plt.xlabel("Predicted cluster")
plt.ylabel("True label")
plt.title(f"Confusion Matrix: {best_rep} + {best_method}")
plt.tight_layout()
plt.savefig("../results/figures/h3_latent_clustering/best_confusion_matrix.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
def show_cluster_prototypes_from_latent(
    X_images,      # (N, 784) or (N, 28, 28)
    Z_latent,      # (N, d), e.g. VAE-16 mu or PCA-16
    y_pred,        # (N,)
    centers,       # (K, d)
    title="",
    save_path=None,
):
    import numpy as np
    import matplotlib.pyplot as plt

    n_clusters = centers.shape[0]
    fig, axes = plt.subplots(1, n_clusters, figsize=(1.8 * n_clusters, 2.2))

    if n_clusters == 1:
        axes = [axes]

    for k in range(n_clusters):
        idx_k = np.where(y_pred == k)[0]
        if len(idx_k) == 0:
            axes[k].axis("off")
            continue

        Z_k = Z_latent[idx_k]                  # 在 latent 空间选该簇样本
        dists = np.sum((Z_k - centers[k])**2, axis=1)
        best_local = np.argmin(dists)
        best_idx = idx_k[best_local]          # 对应到原始样本索引

        img = X_images[best_idx]
        if img.ndim == 1:
            img = img.reshape(28, 28)

        axes[k].imshow(img, cmap="gray")
        axes[k].set_title(f"c{k}")
        axes[k].axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    if save_path is not None:
        plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()

In [ ]:
km_model_vae, km_pred_vae, km_metrics_vae = run_kmeans_eval(
    Z_train_vae16, Z_test_vae16, y_test, n_clusters=10, random_state=42
)

show_cluster_prototypes_from_latent(
    X_images=X_test_raw,
    Z_latent=Z_test_vae16,
    y_pred=km_pred_vae,
    centers=km_model_vae.cluster_centers_,
    title="Cluster prototypes from VAE-16(mu) + KMeans",
    save_path="../results/figures/h3_latent_clustering/vae16_kmeans_prototypes.png"
)

In [ ]:
km_model_pca, km_pred_pca, km_metrics_pca = run_kmeans_eval(
    Z_train_pca16, Z_test_pca16, y_test, n_clusters=10, random_state=42
)

show_cluster_prototypes_from_latent(
    X_images=X_test_raw,
    Z_latent=Z_test_pca16,
    y_pred=km_pred_pca,
    centers=km_model_pca.cluster_centers_,
    title="Cluster prototypes from PCA-16 + KMeans",
    save_path="../results/figures/h3_latent_clustering/pca16_kmeans_prototypes.png"
)

In [ ]:
vae2_ckpt_path = "../results/logs/h1_conv_vae_best_latent2.pt"
use_vae2 = os.path.exists(vae2_ckpt_path)
print("Found VAE-2 checkpoint:", use_vae2)

In [ ]:
if use_vae2:
    vae2 = ConvVAE(
        input_channels=1,
        latent_dim=2,
        hidden_dim=best_config["hidden_dim"],
        base_channels=best_config["base_channels"],
    ).to(device)

    vae2.load_state_dict(torch.load(vae2_ckpt_path, map_location=device))
    vae2.eval()

    enc_test_vae2 = encode_dataset(vae2, test_loader, device=device, use_mu=True)
    Z_test_vae2 = enc_test_vae2["mu"]
    y_test_vae2 = enc_test_vae2["y"]

    km2 = KMeans(n_clusters=10, random_state=42, n_init=20)
    pred2 = km2.fit_predict(Z_test_vae2)

    plt.figure(figsize=(7, 6))
    scatter = plt.scatter(
        Z_test_vae2[:, 0],
        Z_test_vae2[:, 1],
        c=pred2,
        s=8,
        alpha=0.7,
    )
    plt.colorbar(scatter)
    plt.xlabel("z1")
    plt.ylabel("z2")
    plt.title("KMeans clusters on VAE-2 latent means")
    plt.tight_layout()
    plt.savefig("../results/figures/h3_latent_clustering/vae2_kmeans_scatter.png", dpi=200, bbox_inches="tight")
    plt.show()

In [ ]:
summary_lines = [
    "H3 结论草稿：",
    "1. 不同表示空间会显著影响聚类结果，说明表示学习质量对无监督任务非常关键。",
    "2. 原始像素空间虽然包含完整信息，但维数高且局部扰动较强，聚类效果通常不稳定。",
    "3. PCA-16 通过线性降维去除了部分冗余，通常能优于 raw pixels。",
    "4. VAE-16(mu) 进一步学习到更贴近数字流形的非线性潜表示，因此往往在 ARI / NMI / Purity 上表现最好。",
    "5. 这说明卷积 VAE 不仅能做生成与重构，也能为 KMeans / GMM 提供更有利的表征空间。",
]
print("\n".join(summary_lines))

In [ ]:
pca_var = np.var(Z_test_pca16, axis=0)
vae_var = np.var(Z_test_vae16, axis=0)

plt.figure(figsize=(7,4))
plt.plot(np.arange(1, 17), pca_var, marker="o", label="PCA-16 variance")
plt.plot(np.arange(1, 17), vae_var, marker="s", label="VAE-16(mu) variance")
plt.xlabel("Dimension")
plt.ylabel("Variance")
plt.title("Variance profile of latent representations")
plt.legend()
plt.tight_layout()
plt.show()